# 04 — Evaluation & Analysis

Deep-dive vào kết quả sau khi train xong notebook 03.

| Section | Nội dung |
|---------|----------|
| 1 | Load models + data |
| 2 | Per-KC AUC breakdown |
| 3 | Sequence position analysis (early vs late) |
| 4 | Difficulty-stratified performance |
| 5 | User-type analysis (beginner / intermediate / advanced) |
| 6 | Cold-start score vs model accuracy |
| 7 | Best model error analysis |
| 8 | Summary & thesis takeaways |

In [ ]:
import sys, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             mean_squared_error, confusion_matrix,
                             ConfusionMatrixDisplay)

# Kaggle-aware path detection
ON_KAGGLE = Path('/kaggle').exists()

if ON_KAGGLE:
    KG_DATASET = Path('/kaggle/input/datasets/zakhim/rcmsys')
    sys.path.insert(0, str(KG_DATASET))
    PROC     = Path('/kaggle/input/notebooks/zakhim/02-feature-engineering-ipynb/proj/data/processed')
    MODELS   = Path('/kaggle/input/notebooks/zakhim/03-train-models-ipynb/results/models')
    ABLATION = Path('/kaggle/input/notebooks/zakhim/03-train-models-ipynb/results/ablation')
    RAW_SIM  = KG_DATASET / 'data' / 'raw' / 'simulation'
    PLOTS    = Path('/kaggle/working/plots')
else:
    ROOT     = Path().resolve().parent
    sys.path.insert(0, str(ROOT))
    PROC     = ROOT / 'data' / 'processed'
    MODELS   = ROOT / 'results' / 'models'
    ABLATION = ROOT / 'results' / 'ablation'
    RAW_SIM  = ROOT / 'data' / 'raw' / 'simulation'
    PLOTS    = ROOT / 'results' / 'plots'

PLOTS.mkdir(parents=True, exist_ok=True)

from kt_models.bkt  import BKT
from kt_models.dkt  import DKT
from kt_models.sakt import SAKT

sns.set_theme(style='whitegrid', palette='tab10')
print('ON_KAGGLE:', ON_KAGGLE)
print('PROC   :', PROC)
print('MODELS :', MODELS)
print('ABLATION:', ABLATION)
print('RAW_SIM:', RAW_SIM)
print('Setup OK')


In [ ]:
# Load data
with open(PROC / 'metadata.json') as f: meta = json.load(f)
with open(PROC / 'train_seqs.pkl','rb') as f: train_seqs = pickle.load(f)
with open(PROC / 'val_seqs.pkl','rb') as f:   val_seqs   = pickle.load(f)
with open(PROC / 'test_seqs.pkl','rb') as f:  test_seqs  = pickle.load(f)
with open(PROC / 'coldstart.pkl','rb') as f:  coldstart  = pickle.load(f)

N_KCS    = meta['n_kcs']
KC_NAMES = meta['kc_ids']

df_log = pd.read_csv(PROC / 'interactions_featured.csv')

with open(RAW_SIM / 'virtual_users.json') as f:
    users_raw = json.load(f)
user_type_map = {u['user_id']: u['user_type'] for u in users_raw}

print('n_kcs={}, test_users={}'.format(N_KCS, len(test_seqs)))
print('KC names:', KC_NAMES)


In [ ]:
# Load trained models từ results/models/
bkt = BKT(n_kcs=N_KCS).load(str(MODELS / 'bkt.npy'))

dkt_bin = DKT(n_kcs=N_KCS, mode='binary').load(str(MODELS / 'dkt_binary.pt'))
dkt_q   = DKT(n_kcs=N_KCS, mode='quality').load(str(MODELS / 'dkt_quality.pt'))
dkt_deep= DKT(n_kcs=N_KCS, mode='binary').load(str(MODELS / 'dkt_deep.pt'))

sakt_bin = SAKT(n_kcs=N_KCS, mode='binary').load(str(MODELS / 'sakt_binary.pt'))
sakt_q   = SAKT(n_kcs=N_KCS, mode='quality').load(str(MODELS / 'sakt_quality.pt'))
sakt_deep= SAKT(n_kcs=N_KCS, mode='binary').load(str(MODELS / 'sakt_deep.pt'))

# Helper
def prauc(yt, yp):
    try: return round(float(average_precision_score(yt, yp)), 4)
    except: return 0.0

MODELS_EVAL = {
    'BKT':          bkt,
    'DKT-Binary':   dkt_bin,
    'DKT-Quality★': dkt_q,
    'DKT-Deep':     dkt_deep,
    'SAKT-Binary':  sakt_bin,
    'SAKT-Quality★':sakt_q,
    'SAKT-Deep':    sakt_deep,
}
print('Models loaded:', list(MODELS_EVAL.keys()))

In [ ]:
# === Section 2: Per-KC AUC breakdown ===
# Dùng best model: DKT-Quality★ (highest ROC-AUC 0.6720)

def per_kc_auc(model, seqs, n_kcs):
    """Tính AUC riêng cho từng KC trên test_seqs."""
    kc_yt = {k: [] for k in range(n_kcs)}
    kc_yp = {k: [] for k in range(n_kcs)}

    if hasattr(model, 'params'):  # BKT
        for seq in seqs:
            p_L = {k: model._sig(model.params[k][0]) for k in range(n_kcs)}
            EPS = 1e-6
            for kc, correct in zip(seq['kc_seq'], seq['correct_seq']):
                L0,T,S,G = model._unpack(model.params[kc])
                p_c = np.clip(p_L[kc]*(1-S) + (1-p_L[kc])*G, EPS, 1-EPS)
                kc_yt[kc].append(int(correct >= 1))
                kc_yp[kc].append(float(p_c))
                y = int(correct >= 1)
                p_Lg = (p_L[kc]*(1-S)/p_c) if y else (p_L[kc]*S/(1-p_c))
                p_L[kc] = np.clip(p_Lg, EPS, 1-EPS) + (1-np.clip(p_Lg, EPS, 1-EPS))*T
    else:  # DKT / SAKT
        from torch.utils.data import DataLoader
        import torch
        model.model.eval()
        loader = model._loader(seqs)
        # Need per-step KC labels — use df_log instead
        # Simpler: collect predictions then match by position
        # Use flat approach via predict() + kc labels from seqs
        step_kc, step_yt, step_yp = [], [], []
        yt_all, yp_all = model.predict(seqs)
        idx = 0
        for seq in seqs:
            T = min(len(seq['kc_seq'])-1, 200)
            for t in range(T):
                step_kc.append(seq['kc_seq'][t+1])
                step_yt.append(int(seq['correct_seq'][t+1] >= 1))
            idx += T
        for k, yt, yp in zip(step_kc, step_yt, yp_all):
            kc_yt[k].append(yt)
            kc_yp[k].append(float(yp))

    results = {}
    for k in range(n_kcs):
        yt, yp = np.array(kc_yt[k]), np.array(kc_yp[k])
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            results[k] = None
        else:
            results[k] = round(float(roc_auc_score(yt, yp)), 4)
    return results

print('Computing per-KC AUC for BKT and DKT-Quality★...')
kc_auc_bkt  = per_kc_auc(bkt,   test_seqs, N_KCS)
kc_auc_dktq = per_kc_auc(dkt_q, test_seqs, N_KCS)

# Plot
x = np.arange(N_KCS)
w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
bkt_vals  = [kc_auc_bkt.get(k) or 0  for k in range(N_KCS)]
dktq_vals = [kc_auc_dktq.get(k) or 0 for k in range(N_KCS)]
ax.bar(x - w/2, bkt_vals,  w, label='BKT',          color='#4C72B0')
ax.bar(x + w/2, dktq_vals, w, label='DKT-Quality★', color='#DD8452')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random')
ax.set_xticks(x)
ax.set_xticklabels([n.replace('_',' ') for n in KC_NAMES], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC')
ax.set_title('Per-KC AUC — BKT vs DKT-Quality★ (Test Set)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(PLOTS / 'per_kc_auc.png', dpi=150)
plt.show()
print('Saved → results/plots/per_kc_auc.png')

# Print table
kc_df = pd.DataFrame({'KC': KC_NAMES,
                       'n_test': [len([v for v in list(kc_auc_bkt.values())]) for _ in range(N_KCS)],
                       'BKT_AUC': bkt_vals,
                       'DKT_Q_AUC': dktq_vals})
kc_df['delta'] = (kc_df['DKT_Q_AUC'] - kc_df['BKT_AUC']).round(4)
print(kc_df.sort_values('BKT_AUC').to_string(index=False))

In [ ]:
# === Section 3: Sequence Position Analysis ===
# Accuracy theo vị trí bước trong sequence (early vs late)
# Câu hỏi: model có predict tốt hơn khi đã thấy nhiều history chưa?

def position_accuracy(model, seqs, n_bins=8):
    """Chia sequence thành n_bins theo vị trí tương đối."""
    bin_yt = [[] for _ in range(n_bins)]
    bin_yp = [[] for _ in range(n_bins)]

    yt_all, yp_all = model.predict(seqs)
    idx = 0
    for seq in seqs:
        T = min(len(seq['kc_seq'])-1, 200)
        if T < 2: idx += T; continue
        for t in range(T):
            b = min(int(t / T * n_bins), n_bins-1)
            bin_yt[b].append(int(seq['correct_seq'][t+1] >= 1))
            bin_yp[b].append(float(yp_all[idx+t]))
        idx += T

    aucs = []
    for b in range(n_bins):
        yt, yp = np.array(bin_yt[b]), np.array(bin_yp[b])
        if len(np.unique(yt)) < 2:
            aucs.append(None)
        else:
            aucs.append(roc_auc_score(yt, yp))
    return aucs

N_BINS = 8
bin_labels = [f'{int(i*100/N_BINS)}–{int((i+1)*100/N_BINS)}%' for i in range(N_BINS)]

pos_bkt  = position_accuracy(bkt,   test_seqs, N_BINS)
pos_dktq = position_accuracy(dkt_q, test_seqs, N_BINS)
pos_saktd= position_accuracy(sakt_deep, test_seqs, N_BINS)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_BINS)
for vals, name, color in [
    (pos_bkt,  'BKT',        '#4C72B0'),
    (pos_dktq, 'DKT-Quality★','#DD8452'),
    (pos_saktd,'SAKT-Deep',   '#55A868'),
]:
    valid = [(i, v) for i, v in enumerate(vals) if v is not None]
    xi, yi = zip(*valid) if valid else ([], [])
    ax.plot(xi, yi, 'o-', label=name, color=color, linewidth=2)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=30)
ax.set_xlabel('Vị trí trong sequence (% của tổng độ dài)')
ax.set_ylabel('ROC-AUC')
ax.set_title('AUC theo Vị Trí Sequence — Early vs Late', fontweight='bold')
ax.legend()
ax.set_ylim(0.4, 0.9)
plt.tight_layout()
plt.savefig(PLOTS / 'position_auc.png', dpi=150)
plt.show()

In [ ]:
# === Section 4: Difficulty-Stratified Performance ===

def difficulty_auc(model, seqs):
    """AUC theo difficulty (EASY/MEDIUM/HARD)."""
    diff_yt = {0: [], 1: [], 2: []}
    diff_yp = {0: [], 1: [], 2: []}

    yt_all, yp_all = model.predict(seqs)
    idx = 0
    for seq in seqs:
        T = min(len(seq['kc_seq'])-1, 200)
        for t in range(T):
            d = seq['diff_seq'][t+1]
            diff_yt[d].append(int(seq['correct_seq'][t+1] >= 1))
            diff_yp[d].append(float(yp_all[idx+t]))
        idx += T

    result = {}
    for d, name in [(0,'EASY'),(1,'MEDIUM'),(2,'HARD')]:
        yt, yp = np.array(diff_yt[d]), np.array(diff_yp[d])
        if len(np.unique(yt)) < 2:
            result[name] = None
        else:
            result[name] = round(roc_auc_score(yt, yp), 4)
    return result

diff_results = {}
for name, model in [('BKT',bkt),('DKT-Quality★',dkt_q),('SAKT-Deep',sakt_deep)]:
    diff_results[name] = difficulty_auc(model, test_seqs)

diff_df = pd.DataFrame(diff_results).T
diff_df.index.name = 'Model'
print('=== AUC by Difficulty ===')
print(diff_df.round(4).to_string())

# Plot
diff_df.plot(kind='bar', figsize=(9,4), rot=0,
             color=['#55A868','#4C72B0','#C44E52'],
             edgecolor='white')
plt.title('AUC by Difficulty Level', fontweight='bold')
plt.ylabel('ROC-AUC')
plt.ylim(0.4, 0.9)
plt.legend(title='Difficulty')
plt.tight_layout()
plt.savefig(PLOTS / 'difficulty_auc.png', dpi=150)
plt.show()

In [ ]:
# === Section 5: User-Type Analysis ===
# beginner / intermediate / advanced

def usertype_auc(model, seqs, user_type_map):
    type_yt = {'beginner':[],'intermediate':[],'advanced':[]}
    type_yp = {'beginner':[],'intermediate':[],'advanced':[]}

    yt_all, yp_all = model.predict(seqs)
    idx = 0
    for seq in seqs:
        T = min(len(seq['kc_seq'])-1, 200)
        utype = user_type_map.get(seq['user_id'], 'intermediate')
        for t in range(T):
            type_yt[utype].append(int(seq['correct_seq'][t+1] >= 1))
            type_yp[utype].append(float(yp_all[idx+t]))
        idx += T

    result = {}
    for utype in ['beginner','intermediate','advanced']:
        yt, yp = np.array(type_yt[utype]), np.array(type_yp[utype])
        if len(np.unique(yt)) < 2:
            result[utype] = None
        else:
            result[utype] = round(roc_auc_score(yt, yp), 4)
    return result

utype_results = {}
for name, model in [('BKT',bkt),('DKT-Quality★',dkt_q),('SAKT-Deep',sakt_deep)]:
    utype_results[name] = usertype_auc(model, test_seqs, user_type_map)

utype_df = pd.DataFrame(utype_results).T
utype_df.index.name = 'Model'
print('=== AUC by User Type ===')
print(utype_df.round(4).to_string())

utype_df.plot(kind='bar', figsize=(9,4), rot=0,
              color=['#4C72B0','#DD8452','#55A868'],
              edgecolor='white')
plt.title('AUC by User Type (beginner / intermediate / advanced)', fontweight='bold')
plt.ylabel('ROC-AUC')
plt.ylim(0.4, 0.9)
plt.legend(title='User Type')
plt.tight_layout()
plt.savefig(PLOTS / 'usertype_auc.png', dpi=150)
plt.show()

In [ ]:
# === Section 6: Cold-Start Score vs Model Accuracy ===
# Hypothesis: user có cold-start vector tốt hơn → model predict chính xác hơn

def per_user_acc(model, seqs):
    """Accuracy per user."""
    yt_all, yp_all = model.predict(seqs)
    user_acc = {}
    idx = 0
    for seq in seqs:
        T = min(len(seq['kc_seq'])-1, 200)
        yt = np.array([int(seq['correct_seq'][t+1]>=1) for t in range(T)])
        yp = yp_all[idx:idx+T]
        if len(np.unique(yt)) > 1:
            user_acc[seq['user_id']] = roc_auc_score(yt, yp)
        idx += T
    return user_acc

user_acc_dktq = per_user_acc(dkt_q, test_seqs)

# Cold-start quality = mean of cold-start vector (proxy for skill level)
cs_score = {uid: np.mean(vec) for uid, vec in coldstart.items()}

# Merge
cs_acc_data = [(cs_score[uid], acc)
               for uid, acc in user_acc_dktq.items()
               if uid in cs_score]
cs_vals, acc_vals = zip(*cs_acc_data)

corr = np.corrcoef(cs_vals, acc_vals)[0,1]
print(f'Correlation (cold-start mean vs per-user AUC): r={corr:.4f}')

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(cs_vals, acc_vals, alpha=0.5, s=20, color='#4C72B0')
z = np.polyfit(cs_vals, acc_vals, 1)
p = np.poly1d(z)
x_line = np.linspace(min(cs_vals), max(cs_vals), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, label=f'r={corr:.3f}')
ax.set_xlabel('Cold-Start Score (mean self-rating)')
ax.set_ylabel('Per-User ROC-AUC (DKT-Quality★)')
ax.set_title('Cold-Start Score vs Model Accuracy', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'coldstart_vs_acc.png', dpi=150)
plt.show()

In [ ]:
# === Section 7: Error Analysis — Confusion Matrix ===
# Dùng DKT-Quality★ (best ROC-AUC neural model)

yt_all, yp_all = dkt_q.predict(test_seqs)
y_pred_bin = (yp_all >= 0.5).astype(int)

cm = confusion_matrix(yt_all, y_pred_bin)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
disp = ConfusionMatrixDisplay(cm, display_labels=['Fail','Pass'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — DKT-Quality★', fontweight='bold')

# Prediction score distribution
axes[1].hist(yp_all[yt_all==0], bins=40, alpha=0.6, color='#C44E52', label='True Fail')
axes[1].hist(yp_all[yt_all==1], bins=40, alpha=0.6, color='#4C72B0', label='True Pass')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Prediction Distribution by True Label', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS / 'error_analysis.png', dpi=150)
plt.show()

# Metrics
tn, fp, fn, tp = cm.ravel()
precision = tp/(tp+fp) if (tp+fp)>0 else 0
recall    = tp/(tp+fn) if (tp+fn)>0 else 0
f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
print(f'Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}')
print(f'False Positive Rate={fp/(fp+tn):.4f} (predict Pass nhưng thực ra Fail)')
print(f'False Negative Rate={fn/(fn+tp):.4f} (predict Fail nhưng thực ra Pass)')

In [ ]:
# === Section 8: Summary Table ===

ablation = pd.read_csv(ABLATION / 'ablation_results.csv')
ablation['PR-AUC Gain'] = (ablation['test_prauc'] - ablation.loc[ablation['model']=='Baseline','test_prauc'].values[0]).round(4)

summary_cols = ['model','test_auc','test_prauc','PR-AUC Gain','test_rmse','test_acc','elapsed_s']
summary_cols = [c for c in summary_cols if c in ablation.columns]
print('=== Final Summary ===')
print(ablation[summary_cols].sort_values('test_auc', ascending=False).to_string(index=False))

print('\n=== Thesis Takeaways ===')
best = ablation.loc[ablation['test_auc'].idxmax()]
print(f'1. Best ROC-AUC: {best["model"]} ({best["test_auc"]:.4f})')
print(f'2. Quality label giúp DKT: DKT-Quality★ > DKT-Binary ({ablation.loc[ablation.model=="DKT-Quality★","test_auc"].values[0]:.4f} vs {ablation.loc[ablation.model=="DKT-Binary","test_auc"].values[0]:.4f})')
print(f'3. Deep models không cải thiện đáng kể → data size bottleneck')
print(f'4. BKT competitive → simulation bias (per-KC parameterization)')
print(f'5. Tất cả models >> Baseline (AUC 0.5) → KT pipeline có giá trị')